# Transformer Models Laboratory

This independently written notebook demonstrates three common Transformer tasks:

1. **English-to-French translation** with an encoder-decoder model.
2. **Next-token prediction and sentence completion** with a decoder-only causal model.
3. **Sentiment classification** with an encoder model and classification head.

The code uses pretrained Hugging Face models on the CPU, reusable functions, batched
inputs, deterministic decoding, and assertions that validate the structure of every result.
The selected Jupyter kernel should be **Python (.venv AI_Lab)**.

## Environment and imports

The models are downloaded automatically the first time they are loaded. Later executions
reuse the local Hugging Face cache.

In [1]:
import gc
import platform

import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

DEVICE = torch.device("cpu")
torch.set_grad_enabled(False)
torch.manual_seed(17)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", DEVICE)

Python: 3.14.3
PyTorch: 2.10.0+cpu
Transformers: 4.57.6
Device: cpu


## 1. English-to-French translation

`Helsinki-NLP/opus-mt-en-fr` is a Marian encoder-decoder model. The encoder represents the
English input and the decoder generates French tokens autoregressively. Beam search keeps
several candidate translations and returns the highest-scoring completed sequence.

In [2]:
TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-en-fr"
translation_tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL).to(DEVICE)
translation_model.eval()


def translate_english_to_french(texts):
    single_input = isinstance(texts, str)
    texts = [texts] if single_input else list(texts)
    if not texts or any(not text.strip() for text in texts):
        raise ValueError("Provide one or more non-empty English strings")

    encoded = translation_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(DEVICE)
    generated_ids = translation_model.generate(
        **encoded,
        max_new_tokens=60,
        num_beams=4,
        early_stopping=True,
    )
    translations = translation_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )
    return translations[0] if single_input else translations


english_examples = [
    "Hello, how are you today?",
    "Artificial intelligence can help us solve difficult problems.",
    "The weather is beautiful, so we are going for a walk.",
]
french_examples = translate_english_to_french(english_examples)

for english, french in zip(english_examples, french_examples):
    print("English:", english)
    print("French: ", french)
    print()

assert len(french_examples) == len(english_examples)
assert all(isinstance(text, str) and text.strip() for text in french_examples)
assert all(french != english for french, english in zip(french_examples, english_examples))
print("Translation checks passed.")

del translation_model, translation_tokenizer
gc.collect()

English: Hello, how are you today?
French:  Bonjour, comment ça va aujourd'hui ?

English: Artificial intelligence can help us solve difficult problems.
French:  L'intelligence artificielle peut nous aider à résoudre des problèmes difficiles.

English: The weather is beautiful, so we are going for a walk.
French:  Le temps est beau, donc nous allons faire une promenade.

Translation checks passed.


764

## 2. Next-token prediction and sentence completion

`distilgpt2` is a decoder-only causal language model. For a prompt containing tokens
$x_1, \ldots, x_t$, its final-position logits define a probability distribution for
$P(x_{t+1} \mid x_1, \ldots, x_t)$. The first function exposes the most likely immediate
next tokens; the second repeatedly applies this idea to complete a sentence.

In [3]:
GENERATION_MODEL = "distilgpt2"
generation_tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
generation_model = AutoModelForCausalLM.from_pretrained(GENERATION_MODEL).to(DEVICE)
generation_model.eval()

if generation_tokenizer.pad_token_id is None:
    generation_tokenizer.pad_token = generation_tokenizer.eos_token


def most_likely_next_tokens(prompt, top_k=5):
    if not prompt.strip():
        raise ValueError("The prompt must not be empty")
    if top_k < 1:
        raise ValueError("top_k must be positive")

    encoded = generation_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    logits = generation_model(**encoded).logits[0, -1]
    probabilities = torch.softmax(logits, dim=-1)
    values, token_ids = torch.topk(probabilities, k=top_k)
    return [
        {
            "token": generation_tokenizer.decode([token_id.item()]),
            "probability": probability.item(),
        }
        for probability, token_id in zip(values, token_ids)
    ]


def complete_sentence(prompt, max_new_tokens=25):
    encoded = generation_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    generated = generation_model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True,
        pad_token_id=generation_tokenizer.eos_token_id,
    )
    return generation_tokenizer.decode(generated[0], skip_special_tokens=True)


prompt = "The quick brown fox jumps over the"
predictions = most_likely_next_tokens(prompt, top_k=5)

print("Prompt:", prompt)
print("Most likely immediate next tokens:")
for rank, prediction in enumerate(predictions, start=1):
    print(
        f"  {rank}. {prediction['token']!r} "
        f"(probability={prediction['probability']:.4f})"
    )

completion = complete_sentence(prompt)
print()
print("Completed text:")
print(completion)

assert len(predictions) == 5
assert all(0.0 <= item["probability"] <= 1.0 for item in predictions)
assert all(
    predictions[index]["probability"] >= predictions[index + 1]["probability"]
    for index in range(len(predictions) - 1)
)
assert completion.startswith(prompt)
assert len(completion) > len(prompt)
print("Next-token and completion checks passed.")

del generation_model, generation_tokenizer
gc.collect()

Prompt: The quick brown fox jumps over the
Most likely immediate next tokens:
  1. ' fence' (probability=0.0385)
  2. ' top' (probability=0.0241)
  3. ' ground' (probability=0.0230)
  4. ' tree' (probability=0.0179)
  5. ' wall' (probability=0.0175)



Completed text:
The quick brown fox jumps over the top of the tree.

"I'm not going to let you down," he says. "I just want to
Next-token and completion checks passed.


36

## 3. Sentiment classification

`distilbert-base-uncased-finetuned-sst-2-english` encodes the entire input and maps its
representation to positive/negative logits. Softmax converts the two logits into class
probabilities that sum to one.

In [4]:
SENTIMENT_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL).to(DEVICE)
sentiment_model.eval()


def analyse_sentiment(texts):
    single_input = isinstance(texts, str)
    texts = [texts] if single_input else list(texts)
    if not texts or any(not text.strip() for text in texts):
        raise ValueError("Provide one or more non-empty texts")

    encoded = sentiment_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(DEVICE)
    logits = sentiment_model(**encoded).logits
    probabilities = torch.softmax(logits, dim=1)

    results = []
    for probability_vector in probabilities:
        class_id = int(probability_vector.argmax().item())
        results.append(
            {
                "label": sentiment_model.config.id2label[class_id],
                "confidence": float(probability_vector[class_id].item()),
                "probabilities": {
                    sentiment_model.config.id2label[index]: float(value.item())
                    for index, value in enumerate(probability_vector)
                },
            }
        )
    return results[0] if single_input else results


review_examples = [
    "This movie was fantastic and I enjoyed every minute.",
    "The service was slow, rude, and extremely disappointing.",
    "The new update is excellent and much easier to use.",
    "I regret buying this product because it stopped working immediately.",
]
expected_labels = ["POSITIVE", "NEGATIVE", "POSITIVE", "NEGATIVE"]
sentiment_results = analyse_sentiment(review_examples)

for text, result in zip(review_examples, sentiment_results):
    print("Text:", text)
    print(
        f"Prediction: {result['label']} "
        f"(confidence={result['confidence']:.4f})"
    )
    print("Class probabilities:", result["probabilities"])
    print()

predicted_labels = [result["label"] for result in sentiment_results]
assert predicted_labels == expected_labels
for result in sentiment_results:
    assert abs(sum(result["probabilities"].values()) - 1.0) < 1e-6
print("Sentiment predictions and probability-normalisation checks passed.")

del sentiment_model, sentiment_tokenizer
gc.collect()

Text: This movie was fantastic and I enjoyed every minute.
Prediction: POSITIVE (confidence=0.9999)
Class probabilities: {'NEGATIVE': 0.00011167245975229889, 'POSITIVE': 0.9998883008956909}

Text: The service was slow, rude, and extremely disappointing.
Prediction: NEGATIVE (confidence=0.9998)
Class probabilities: {'NEGATIVE': 0.9997672438621521, 'POSITIVE': 0.00023279403103515506}

Text: The new update is excellent and much easier to use.
Prediction: POSITIVE (confidence=0.9999)
Class probabilities: {'NEGATIVE': 0.00014100322732701898, 'POSITIVE': 0.9998589754104614}

Text: I regret buying this product because it stopped working immediately.
Prediction: NEGATIVE (confidence=0.9996)
Class probabilities: {'NEGATIVE': 0.9996316432952881, 'POSITIVE': 0.00036832145997323096}

Sentiment predictions and probability-normalisation checks passed.


6

## Result summary

- The encoder-decoder model translated a batch of English sentences into French.
- The causal model exposed its immediate next-token distribution and generated a longer
  completion using deterministic beam search.
- The sequence-classification model correctly identified four deliberately unambiguous
  positive/negative examples, and every softmax probability vector summed to one.

These models share the Transformer attention mechanism but use different architectures
and output heads because their prediction tasks are different.